# UD4.03 — Seaborn: gráficos estadísticos en una línea

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
UD4 — Visualización de datos · 14 horas

Criterios 1.d y 2.e · Material de partida de las prácticas P4.1 y P4.2

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

- **Situar Seaborn respecto a Matplotlib** y explicar qué es cada una
- **Distinguir funciones de nivel `Axes` y de nivel figura**, que es el concepto que
  más errores causa
- **Elegir tema, contexto y paleta** según dónde va a verse el gráfico
- **Dibujar distribuciones, categorías y relaciones** con la API actual de Seaborn
- **Usar `FacetGrid` y `pairplot`** para mirar muchas variables a la vez
- **Justificar con líneas de código medidas** cuándo conviene Seaborn y cuándo no
- **Reconocer lo que Seaborn calcula por ti** y por qué eso puede ser un problema

## 1. Qué es Seaborn

Seaborn **no sustituye a Matplotlib: se apoya en ella**. Cuando llamas a
`sns.boxplot(...)`, Seaborn calcula los cuartiles y después llama a métodos de un
`Axes` de Matplotlib para dibujarlos. La prueba es que devuelve ese mismo `Axes`, y
que todo lo del cuaderno 01 sigue funcionando encima.

Lo que aporta:

| Aporta | En concreto |
|---|---|
| **Estadística incluida** | Cuartiles, densidades, intervalos de confianza y regresiones, calculados solos |
| **Hablar de columnas** | `x="Categoria"` en lugar de tener que extraer arrays |
| **Agrupar por una variable** | `hue=`, `col=` y `row=` reparten los datos sin bucles |
| **Paletas y temas decentes** | Sin tener que configurar nada |

Lo que **no** aporta: control fino. Cuando hay que colocar una anotación en un sitio
exacto o ajustar un eje, se vuelve a Matplotlib sobre el `Axes` que Seaborn ha
devuelto. Las dos bibliotecas se usan juntas, no una o la otra.

In [ ]:
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

rng = np.random.default_rng(20262027)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# `set_theme` fija tema, contexto y paleta de golpe. Se hace UNA vez, al principio del
# cuaderno, y a partir de aquí todos los gráficos lo heredan.
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")

print("seaborn   ", sns.__version__)
print("matplotlib", matplotlib.__version__)
print("pandas    ", pd.__version__)

## 2. Los datos

Seaborn trae conjuntos de ejemplo (`sns.load_dataset("tips")`, `"penguins"`,
`"titanic"`), pero **los descarga de internet cada vez**. Un cuaderno que necesita red
para arrancar es un cuaderno que un día no arranca, así que aquí se trabaja con los
datos del módulo: los de **TechStore**, los mismos de la UD3.

Eso tiene una ventaja de fondo: ya sabes lo que hay dentro, incluidos sus defectos.

In [ ]:
# Solo en Google Colab: descarga el fichero de datos de la unidad.
import os
import urllib.request

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD4/datos/")
FICHERO = "ecommerce_ventas_2024.csv"

if not os.path.exists(os.path.join("datos", FICHERO)):
    os.makedirs("datos", exist_ok=True)
    urllib.request.urlretrieve(BASE + FICHERO, os.path.join("datos", FICHERO))
    print("descargado:", FICHERO)
else:
    print("El fichero ya está en datos/, no descargo nada")

### 2.1 Antes de dibujar, limpiar

Esta sección no es un trámite. **Un gráfico de datos sucios es un gráfico de la
suciedad**, y como los gráficos son convincentes, el error pasa desapercibido mucho
más tiempo que en una tabla.

Vamos a verlo: primero se dibuja el fichero tal cual sale del disco.

In [ ]:
bruto = pd.read_csv(os.path.join("datos", FICHERO))
bruto["Fecha"] = pd.to_datetime(bruto["Fecha"])

print(f"{len(bruto)} transacciones, {bruto.shape[1]} columnas")
print()
print(bruto.dtypes)
print()
print("Y esto es lo que trae dentro:")
print(f"  precios negativos           {(bruto['Precio_Unitario'] < 0).sum():>4}")
print(f"  descuentos fuera de [0,100] "
      f"{(~bruto['Descuento_%'].between(0, 100) & bruto['Descuento_%'].notna()).sum():>4}")
print(f"  cantidades por encima de 50 {(bruto['Cantidad'] > 50).sum():>4}")
print(f"  regiones ausentes           {bruto['Region'].isna().sum():>4}")
print(f"  grafías de Categoria        {bruto['Categoria'].nunique():>4} "
      f"(deberían ser 6)")
print()
print("Las grafías, una a una:")
for grafia, veces in bruto["Categoria"].value_counts().items():
    print(f"  {grafia!r:>20}  {veces}")

In [ ]:
# El mismo gráfico, con los datos sucios y con los datos limpios.
bruto_importe = (bruto["Precio_Unitario"] * bruto["Cantidad"]
                 * (1 - bruto["Descuento_%"].fillna(0) / 100)
                 + bruto["Costo_Envio"])

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(x=bruto_importe, ax=izq, color="#e74c3c")
izq.set_title("Importe por transacción, datos SIN limpiar\n"
              f"El eje llega a {bruto_importe.max():,.0f} € por unas pocas filas",
              fontweight="bold", fontsize=11)
izq.set_xlabel("Importe (€)")

limite = bruto_importe.quantile(0.995)
sns.boxplot(x=bruto_importe[bruto_importe.between(0, limite)], ax=der,
            color="#27ae60")
der.set_title("El mismo dato, sin el 0,5 % superior y sin negativos\n"
              "Ahora se ve la caja, que es de lo que iba el gráfico",
              fontweight="bold", fontsize=11)
der.set_xlabel("Importe (€)")

fig.tight_layout()
plt.show()

print(f"Importe máximo en bruto: {bruto_importe.max():>12,.2f} €")
print(f"Importe mínimo en bruto: {bruto_importe.min():>12,.2f} €")
print(f"Percentil 99,5:          {limite:>12,.2f} €")
print()
print("En el gráfico de la izquierda la caja es una raya. No es que los datos no")
print("tengan estructura: es que doce filas con Cantidad hasta 999 y veintidós con")
print("precio negativo estiran el eje y aplastan al resto.")
print()
print("Y ojo con lo que NO se ve en la izquierda: el precio negativo. Un boxplot con")
print("bigotes hasta 40.000 no deja distinguir si el mínimo es 5 € o -300 €. El")
print("gráfico no ha avisado del error; lo ha escondido.")

### 2.2 La limpieza, documentada

Es la de la UD3, resumida, y con el registro de cuántas filas se van en cada paso.
Ese registro es lo que permite que alguien discuta las decisiones en lugar de
creérselas.

In [ ]:
def limpia(datos):
    """Aplica las reglas del dominio de TechStore y devuelve los datos y el registro.

    Cada regla sale del significado de la columna, no de mirar la distribución:
      - un precio no puede ser negativo,
      - un descuento va de 0 a 100 por ciento,
      - una transacción de más de 50 unidades en una tienda al por menor es un error
        de captura, no un pedido,
      - la región es necesaria para el análisis por zonas, y no se puede imputar.
    """
    registro = [("de partida", len(datos))]
    d = datos.copy()

    # Grafías: espacios de sobra, mayúsculas y tildes inconsistentes. La tilde es la
    # que se olvida siempre: 'telefonia' y 'telefonía' son dos valores distintos para
    # pandas y la misma categoría para cualquier persona.
    d["Categoria"] = (d["Categoria"].str.strip().str.lower()
                      .str.normalize("NFKD")
                      .str.encode("ascii", "ignore").str.decode("ascii"))
    registro.append(("tras normalizar Categoria",
                     len(d), f"{d['Categoria'].nunique()} categorías"))

    for descripcion, mascara in [
            ("precio positivo", d["Precio_Unitario"] > 0),
            ("cantidad entre 1 y 50", d["Cantidad"].between(1, 50)),
            ("descuento entre 0 y 100",
             d["Descuento_%"].between(0, 100) | d["Descuento_%"].isna()),
            ("región conocida", d["Region"].notna())]:
        antes = len(d)
        d = d[mascara.reindex(d.index, fill_value=False)]
        registro.append((descripcion, len(d), f"-{antes - len(d)} filas"))

    # El descuento ausente se imputa a cero, que es lo que significa en este dominio:
    # si no se registró descuento, no hubo descuento. Y se deja constancia con una
    # columna indicadora, que es la norma de la UD3.
    d["Descuento_imputado"] = d["Descuento_%"].isna()
    d["Descuento_%"] = d["Descuento_%"].fillna(0.0)

    # Duplicados con identificador distinto: los que `duplicated()` a secas no ve.
    antes = len(d)
    d = d.drop_duplicates(subset=["Fecha", "Cliente_ID", "Producto",
                                  "Precio_Unitario", "Cantidad"])
    registro.append(("sin duplicados de contenido", len(d), f"-{antes - len(d)} filas"))

    # Columnas derivadas que hacen falta para los gráficos.
    d["Importe"] = (d["Precio_Unitario"] * d["Cantidad"]
                    * (1 - d["Descuento_%"] / 100) + d["Costo_Envio"])
    d["Mes"] = d["Fecha"].dt.to_period("M").dt.to_timestamp()
    d["Dia_semana"] = d["Fecha"].dt.dayofweek
    return d.reset_index(drop=True), registro


ventas, registro = limpia(bruto)

print(f"{'paso':>32}  {'filas':>6}  detalle")
print("-" * 62)
for fila in registro:
    detalle = fila[2] if len(fila) > 2 else ""
    print(f"{fila[0]:>32}  {fila[1]:>6}  {detalle}")

print()
print(f"Se han descartado {len(bruto) - len(ventas)} filas de {len(bruto)} "
      f"({(1 - len(ventas) / len(bruto)) * 100:.1f} %).")
print()
print("Categorías finales:", ", ".join(sorted(ventas["Categoria"].unique())))
print("Regiones:          ", ", ".join(sorted(ventas["Region"].unique())))
print()
assert ventas["Precio_Unitario"].gt(0).all(), "quedan precios no positivos"
assert ventas["Categoria"].nunique() == 6, "las categorías no se han unificado"
assert ventas["Region"].notna().all(), "quedan regiones ausentes"
print("Comprobaciones pasadas: los datos están listos para dibujar.")

ventas.head()

## 3. El concepto que hay que entender: nivel `Axes` y nivel figura

Seaborn tiene dos familias de funciones y confundirlas es el error más común.

### Funciones de nivel `Axes`

`histplot`, `kdeplot`, `boxplot`, `violinplot`, `scatterplot`, `lineplot`, `barplot`,
`heatmap`, `regplot`...

- Dibujan sobre **un `Axes` que le das tú** con `ax=`.
- Devuelven ese `Axes`.
- Se combinan con `plt.subplots` y con Matplotlib puro.

### Funciones de nivel figura

`displot`, `catplot`, `relplot`, `lmplot`, `pairplot`, `jointplot`.

- **Crean su propia figura.** No aceptan `ax=`.
- Devuelven un objeto `FacetGrid` (o `JointGrid`), no un `Axes`.
- A cambio saben hacer **reparto en paneles** con `col=` y `row=`.

### La regla

> Si has escrito `fig, ax = plt.subplots()`, usa una función de nivel `Axes` y pásale
> `ax=`. Si quieres que Seaborn te monte los paneles, usa una de nivel figura y **no
> crees ninguna figura antes**.

El síntoma de haberlo confundido: aparece una figura vacía al lado de la buena. Es
la que creaste tú y Seaborn ignoró.

In [ ]:
# EL ERROR, para verlo una vez: crear la figura y llamar a una función de nivel figura.
print("Lo que pasa al mezclar los dos niveles:")
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.set_title("Esta figura la he creado yo, y se queda vacía")
sns.catplot(data=ventas, x="Categoria", y="Importe", kind="box", height=3, aspect=2.2)
plt.show()

print()
print("Han salido DOS figuras: la vacía que creé con plt.subplots y la que ha creado")
print("catplot por su cuenta. `catplot` no acepta ax= y no hay forma de meterlo")
print("dentro de una figura ajena.")

In [ ]:
# LO CORRECTO, de las dos formas.

# Forma A — nivel Axes: yo monto el lienzo, Seaborn dibuja dentro.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=ventas, x="Categoria", y="Importe",
            hue="Categoria", legend=False, ax=axes[0])
axes[0].set_title("boxplot, nivel Axes, en el panel que yo elijo", fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylim(0, ventas["Importe"].quantile(0.99))

sns.histplot(data=ventas, x="Importe", bins=40, ax=axes[1])
axes[1].set_title("histplot, nivel Axes, en el otro panel", fontweight="bold")
axes[1].set_xlim(0, ventas["Importe"].quantile(0.99))

fig.suptitle("Forma A: nivel Axes. El lienzo lo monto yo", fontweight="bold",
             fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Forma B — nivel figura: Seaborn monta el lienzo y reparte por una variable.
rejilla = sns.catplot(data=ventas, x="Categoria", y="Importe", kind="box",
                      col="Region", col_wrap=4, height=2.8, aspect=1.15,
                      hue="Categoria", legend=False)
rejilla.set_xticklabels(rotation=60, ha="right", fontsize=7)
rejilla.set_titles("{col_name}", fontweight="bold", size=10)
rejilla.set_axis_labels("", "Importe (€)")
rejilla.set(ylim=(0, ventas["Importe"].quantile(0.99)))
rejilla.figure.suptitle("Forma B: nivel figura. Siete paneles y ningún bucle",
                        fontweight="bold", fontsize=13, y=1.03)
plt.show()

print("Los siete paneles del gráfico anterior, con Matplotlib puro, serían un bucle")
print("sobre las regiones, un filtrado por región, el cálculo de los cuartiles y la")
print("gestión de la rejilla. Con `col=\"Region\"` es una palabra.")
print()
print("Y fíjate en `rejilla.figure`: el FacetGrid guarda dentro la figura de")
print("Matplotlib, así que se le puede poner suptitle y guardar con savefig.")
print("En versiones antiguas de Seaborn el atributo se llamaba `.fig`, y en la 0.13")
print("está obsoleto: si un ejemplo de internet usa `g.fig`, es de antes de 2023.")

## 4. Temas, contextos y paletas

Seaborn separa tres decisiones que Matplotlib mezcla:

| Decisión | Qué controla | Valores |
|---|---|---|
| **Tema** (`style`) | Fondo y rejilla | `darkgrid`, `whitegrid`, `dark`, `white`, `ticks` |
| **Contexto** (`context`) | Tamaño de todo: letras, líneas, marcadores | `paper`, `notebook`, `talk`, `poster` |
| **Paleta** (`palette`) | Los colores | `colorblind`, `deep`, `Set2`, `viridis`... |

La separación es útil porque las tres responden a preguntas distintas: el tema, a si
hay que leer valores exactos; el contexto, a **a qué distancia se va a mirar**; la
paleta, a qué tipo de variable va en el color.

### 4.1 El tema

- **`whitegrid`**: fondo blanco, rejilla gris. Es el que hay que usar **cuando hay que
  comparar valores**, porque la rejilla permite llevar la vista de la barra al eje.
  Barras y cajas, casi siempre.
- **`darkgrid`**: fondo gris. Cansa menos la vista en pantalla y en proyector.
- **`white`** y **`ticks`**: sin rejilla. Para papel, donde la rejilla gasta tinta y
  ensucia. `ticks` añade las marquitas en los ejes, que es el estilo clásico de las
  revistas científicas.
- **`dark`**: fondo gris sin rejilla. Para mapas de calor e imágenes, donde la
  rejilla estorba.

In [ ]:
temas = ["whitegrid", "darkgrid", "white", "ticks", "dark"]
muestra = ventas.sample(300, random_state=20262027)

fig = plt.figure(figsize=(15, 8))
fig.suptitle("Los cinco temas de Seaborn, con el mismo gráfico", fontsize=15,
             fontweight="bold")

for i, tema in enumerate(temas, start=1):
    with sns.axes_style(tema):
        ax = fig.add_subplot(2, 3, i)
        sns.scatterplot(data=muestra, x="Precio_Unitario", y="Importe",
                        hue="Categoria", s=25, alpha=0.8, legend=False, ax=ax)
        ax.set_title(f"style='{tema}'", fontweight="bold", fontsize=11)
        ax.set_xlabel("Precio unitario (€)", fontsize=9)
        ax.set_ylabel("Importe (€)", fontsize=9)
        ax.tick_params(labelsize=8)

texto = fig.add_subplot(2, 3, 6)
texto.axis("off")
texto.text(0.0, 0.5,
           "Cuándo usar cada uno\n\n"
           "whitegrid  comparar valores\n"
           "           (barras, cajas)\n\n"
           "darkgrid   pantalla y proyector,\n"
           "           cansa menos\n\n"
           "white      papel, sin rejilla\n\n"
           "ticks      revistas científicas\n\n"
           "dark       mapas de calor\n"
           "           e imágenes",
           fontsize=10, family="monospace", va="center")

fig.tight_layout()
plt.show()

### 4.2 El contexto

El contexto es el que más se olvida y el que más se nota. Escala **todo**
proporcionalmente, así que el mismo código produce un gráfico legible en un folio o
desde el fondo de un aula sin tocar ni un tamaño de letra a mano.

In [ ]:
resumen_region = (ventas.groupby("Region", as_index=False)["Importe"]
                  .mean().sort_values("Importe", ascending=False))

fig = plt.figure(figsize=(14, 9))
fig.suptitle("El mismo código con cuatro contextos", fontsize=15, fontweight="bold")

for i, contexto in enumerate(["paper", "notebook", "talk", "poster"], start=1):
    with sns.plotting_context(contexto):
        ax = fig.add_subplot(2, 2, i)
        sns.barplot(data=resumen_region, x="Region", y="Importe",
                    hue="Region", legend=False, ax=ax)
        ax.set_title(f"context='{contexto}'", fontweight="bold")
        ax.set_ylabel("Importe medio (€)")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=75)

fig.tight_layout()
plt.show()

print("Reglas de uso:")
print("  paper     figura pequeña dentro de un documento")
print("  notebook  trabajo en el cuaderno (el que está puesto por defecto)")
print("  talk      diapositiva o proyector")
print("  poster    un póster que se mira de pie, a un metro")

### 4.3 Las paletas

La decisión es la misma que la del cuaderno 02, con los mismos tres tipos, y con una
regla añadida sobre el uso del color:

> **El color tiene que codificar una variable.** Si no codifica nada, es decoración,
> y la decoración en un gráfico técnico solo añade ruido.

Seaborn lo ha convertido en una norma de su API. En la versión 0.13, pasar `palette=`
**sin** decir qué variable va en el color produce un aviso de obsolescencia, y la
forma correcta es `hue="la_variable"`. Si lo que quieres es solo que las barras sean
de colores distintos, se escribe `hue=` con la misma variable del eje y
`legend=False`, y así queda explícito que el color no aporta información nueva.

In [ ]:
paletas = {
    "Cualitativas": ["colorblind", "deep", "muted", "pastel", "Set2", "tab10"],
    "Secuenciales": ["Blues", "rocket", "mako", "flare", "viridis", "crest"],
    "Divergentes": ["vlag", "icefire", "coolwarm", "RdBu"],
}

alto = sum(len(v) for v in paletas.values())
fig, axes = plt.subplots(alto, 1, figsize=(9, alto * 0.34))
fig.subplots_adjust(hspace=1.4, left=0.26, right=0.98, top=0.93, bottom=0.02)

i = 0
for categoria, nombres in paletas.items():
    for j, nombre in enumerate(nombres):
        ax = axes[i]
        colores = sns.color_palette(nombre, n_colors=9)
        for k, color in enumerate(colores):
            ax.add_patch(plt.Rectangle((k, 0), 0.94, 1, facecolor=color,
                                       edgecolor="white", linewidth=0.8))
        ax.set_xlim(0, 9)
        ax.set_ylim(0, 1)
        ax.axis("off")
        etiqueta = f"{categoria}\n{nombre}" if j == 0 else nombre
        ax.text(-0.15, 0.5, etiqueta, ha="right", va="center", fontsize=8,
                fontweight="bold" if j == 0 else "normal")
        i += 1

fig.suptitle("Paletas de Seaborn", fontsize=13, fontweight="bold")
plt.show()

print("Las tres decisiones, una vez más:")
print()
print("  categorías sin orden        -> cualitativa   ('colorblind')")
print("  una magnitud ordenada       -> secuencial    ('rocket', 'viridis')")
print("  una magnitud con centro     -> divergente    ('vlag', center=0)")
print()
print("`colorblind` como cualitativa por defecto: usa azul y naranja en lugar de")
print("rojo y verde, y por tanto la lee también el 8 % de los hombres que confunde")
print("esos dos colores. Es la que está puesta en el `set_theme` de este cuaderno.")

## 5. Distribuciones

### 5.1 Histograma, densidad y función de distribución

Tres formas de ver un reparto, y la tercera es la más honesta y la que menos se usa.

- **`histplot`**: el histograma. Depende de `bins`, con el problema visto en el
  cuaderno 01.
- **`kdeplot`**: una curva de densidad suavizada. Bonita, y con **dos** parámetros
  ocultos: el ancho de banda y el hecho de que suaviza por los bordes, así que puede
  dibujar densidad donde no hay datos posibles (importes negativos, por ejemplo).
- **`ecdfplot`**: la función de distribución acumulada empírica. **No tiene ningún
  parámetro que ajustar**: dibuja exactamente los datos. Se lee «qué proporción de
  las observaciones está por debajo de este valor», y para comparar dos grupos es
  mejor que las otras dos.

In [ ]:
recorte = ventas[ventas["Importe"] < ventas["Importe"].quantile(0.99)]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Cuatro formas de mirar el reparto del importe", fontsize=15,
             fontweight="bold")

ax = axes[0, 0]
sns.histplot(data=recorte, x="Importe", bins=40, ax=ax)
ax.set_title("histplot: el recuento por intervalo", fontweight="bold")
ax.set_xlabel("Importe (€)")

ax = axes[0, 1]
sns.histplot(data=recorte, x="Importe", bins=40, kde=True, stat="density", ax=ax)
ax.set_title("histplot con kde=True: densidad y curva suavizada", fontweight="bold")
ax.set_xlabel("Importe (€)")

ax = axes[1, 0]
sns.kdeplot(data=recorte, x="Importe", hue="Categoria", fill=True, alpha=0.25,
            linewidth=1.5, ax=ax)
ax.set_title("kdeplot por categoría: seis curvas se solapan y cuesta leerlas",
             fontweight="bold")
ax.set_xlabel("Importe (€)")

ax = axes[1, 1]
sns.ecdfplot(data=recorte, x="Importe", hue="Categoria", linewidth=1.8, ax=ax)
ax.set_title("ecdfplot por categoría: seis curvas que SÍ se leen",
             fontweight="bold")
ax.set_xlabel("Importe (€)")
ax.set_ylabel("Proporción acumulada")

fig.tight_layout()
plt.show()

mediana_por_categoria = ventas.groupby("Categoria")["Importe"].median().sort_values()
print("Cómo se lee el último panel: a la altura 0,5 del eje Y, el valor del eje X")
print("de cada curva es la MEDIANA de esa categoría. Y se leen las seis de un vistazo:")
print()
for categoria, valor in mediana_por_categoria.items():
    print(f"  {categoria:>16}  mediana {valor:>8,.2f} €")
print()
print("La curva más a la izquierda es la categoría más barata, y la de más a la")
print("derecha la más cara. En el panel de las densidades esa lectura es imposible.")

### 5.2 Categorías: caja, violín, puntos

Comparar el reparto de una variable numérica entre categorías. Seaborn tiene seis
funciones para lo mismo y la elección depende de **cuántos datos hay por grupo**:

| Función | Cuándo |
|---|---|
| `boxplot` | Muchos grupos, o hay que comparar medianas de un vistazo |
| `violinplot` | Interesa la forma, y hay más de 50 observaciones por grupo |
| `boxenplot` | Muchos datos y interesan las colas |
| `stripplot` | Pocos datos: se ve cada observación |
| `swarmplot` | Pocos datos, sin que los puntos se tapen |
| `barplot` | **Cuidado**: solo dibuja la media y su intervalo. Esconde la forma |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Seis maneras de comparar el importe entre categorías", fontsize=15,
             fontweight="bold")

orden = list(mediana_por_categoria.index)
tope = ventas["Importe"].quantile(0.98)
graficos = [
    ("boxplot", sns.boxplot, {}),
    ("violinplot", sns.violinplot, {}),
    ("boxenplot", sns.boxenplot, {}),
    ("stripplot", sns.stripplot, {"alpha": 0.35, "size": 2.5}),
    ("barplot (media e IC)", sns.barplot, {"errorbar": ("ci", 95), "capsize": 0.15}),
]

for ax, (nombre, funcion, extra) in zip(axes.ravel(), graficos):
    funcion(data=ventas, x="Categoria", y="Importe", order=orden,
            hue="Categoria", legend=False, ax=ax, **extra)
    ax.set_title(nombre, fontweight="bold")
    ax.set_ylim(0, tope)
    ax.set_xlabel("")
    ax.set_ylabel("Importe (€)")
    ax.tick_params(axis="x", rotation=60, labelsize=8)

# El sexto: violín con los puntos encima, que es la combinación más informativa.
ax = axes[1, 2]
sns.violinplot(data=ventas, x="Categoria", y="Importe", order=orden,
               hue="Categoria", legend=False, inner=None, alpha=0.4, ax=ax)
sns.stripplot(data=ventas.sample(700, random_state=20262027),
              x="Categoria", y="Importe", order=orden,
              color="black", alpha=0.3, size=1.8, ax=ax)
ax.set_title("violinplot + stripplot", fontweight="bold")
ax.set_ylim(0, tope)
ax.set_xlabel("")
ax.set_ylabel("Importe (€)")
ax.tick_params(axis="x", rotation=60, labelsize=8)

fig.tight_layout()
plt.show()

print("El aviso del `barplot`, con números:")
print()
resumen = ventas.groupby("Categoria")["Importe"].agg(["mean", "median", "std", "count"])
print(resumen.round(2).to_string())
print()
print("El barplot dibuja la primera columna. Fíjate en la desviación: en varias")
print("categorías es MAYOR que la media, o sea que el reparto tiene una cola larga y")
print("la media no representa al caso típico. Compara media y mediana en la tabla.")
print()
print("Un barplot de estos datos afirma que cada categoría tiene 'un' importe típico.")
print("El boxplot y el violín muestran que no lo tiene.")

### 5.3 Lo que Seaborn calcula por ti, y por qué hay que saberlo

`barplot` no dibuja los datos: dibuja **una media y un intervalo de confianza que
calcula por remuestreo**. Eso es una decisión estadística, viene puesta por defecto y
casi nadie la mira.

El parámetro es `errorbar`, y admite cuatro cosas distintas que **significan cosas
distintas**:

| `errorbar=` | Qué dibuja | Qué dice |
|---|---|---|
| `("ci", 95)` (por defecto) | Intervalo de confianza al 95 % de la media | Cuánta incertidumbre hay sobre **la media** |
| `"sd"` | Desviación típica | Cuánto **varían los datos** |
| `("pi", 95)` | Intervalo entre percentiles | Dónde está el 95 % **de las observaciones** |
| `None` | Nada | — |

La confusión entre la primera y la segunda es continua, y no es un detalle: con
muchos datos, el intervalo de confianza de la media se hace diminuto aunque los datos
estén muy dispersos. Una barra con un bigote minúsculo **no** significa que los datos
se parezcan entre sí.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.5), sharey=True)
fig.suptitle("El mismo barplot con las cuatro opciones de `errorbar`",
             fontsize=15, fontweight="bold")

opciones = [(("ci", 95), "('ci', 95)\nincertidumbre de la media"),
            ("sd", "'sd'\ndispersión de los datos"),
            (("pi", 95), "('pi', 95)\ndónde está el 95 % de los datos"),
            (None, "None\nsolo la media")]

for ax, (opcion, titulo) in zip(axes, opciones):
    sns.barplot(data=ventas, x="Categoria", y="Importe", order=orden,
                hue="Categoria", legend=False, errorbar=opcion,
                capsize=0.15, err_kws={"linewidth": 1.6}, ax=ax)
    ax.set_title(titulo, fontweight="bold", fontsize=10)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=70, labelsize=7)
axes[0].set_ylabel("Importe medio (€)")

fig.tight_layout()
plt.show()

n_por_grupo = ventas.groupby("Categoria")["Importe"].size()
print(f"{'categoría':>16} {'n':>6} {'media':>9} {'desv.':>9} "
      f"{'error de la media':>18}")
print("-" * 62)
for categoria in orden:
    grupo = ventas.loc[ventas["Categoria"] == categoria, "Importe"]
    error_medio = grupo.std() / np.sqrt(len(grupo))
    print(f"{categoria:>16} {len(grupo):>6} {grupo.mean():>9.2f} "
          f"{grupo.std():>9.2f} {error_medio:>18.2f}")

print()
print("Mira la última columna comparada con la penúltima: el error de la media es")
print("unas treinta veces más pequeño que la desviación, porque hay cientos de")
print("observaciones por grupo y el error de la media va con la raíz de n.")
print()
print("Por eso los bigotes del primer panel son casi invisibles y los del segundo")
print("enormes. Los dos son correctos y responden a preguntas distintas. Poner el")
print("primero y hablar de 'variabilidad' es un error de bulto, y es frecuentísimo.")

## 6. Relaciones entre variables

### 6.1 Dispersión y regresión

`scatterplot` acepta `hue`, `size` y `style` para meter hasta tres variables más.
`regplot` añade una recta de regresión con su banda de confianza, y ahí hay que tener
cuidado: **la recta la calcula sin preguntar y sin comprobar que tenga sentido**. Si
la relación no es lineal, dibuja una recta igual.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Relación entre precio unitario e importe", fontsize=15,
             fontweight="bold")

muestra = ventas.sample(900, random_state=20262027)

ax = axes[0, 0]
sns.scatterplot(data=muestra, x="Precio_Unitario", y="Importe", hue="Categoria",
                s=25, alpha=0.75, ax=ax)
ax.set_title("scatterplot con hue", fontweight="bold")
ax.legend(fontsize=7, title="Categoría", title_fontsize=8)
ax.set_xlabel("Precio unitario (€)")
ax.set_ylabel("Importe (€)")

ax = axes[0, 1]
sns.scatterplot(data=muestra, x="Precio_Unitario", y="Importe", size="Cantidad",
                hue="Cantidad", sizes=(10, 140), palette="rocket", alpha=0.7, ax=ax)
ax.set_title("scatterplot con size y hue en la misma variable", fontweight="bold")
ax.legend(fontsize=7, title="Cantidad", title_fontsize=8)
ax.set_xlabel("Precio unitario (€)")
ax.set_ylabel("Importe (€)")

ax = axes[1, 0]
sns.regplot(data=muestra, x="Precio_Unitario", y="Importe",
            scatter_kws={"alpha": 0.35, "s": 18}, line_kws={"color": "#c0392b"},
            ax=ax)
correlacion = float(muestra[["Precio_Unitario", "Importe"]].corr().iloc[0, 1])
ax.set_title(f"regplot: recta y banda de confianza\n"
             f"correlación = {correlacion:.3f}", fontweight="bold")
ax.set_xlabel("Precio unitario (€)")
ax.set_ylabel("Importe (€)")

# El aviso: la misma función sobre una relación que NO es lineal.
ax = axes[1, 1]
x_curvo = rng.uniform(-3, 3, 400)
y_curvo = x_curvo ** 2 + rng.normal(0, 1.2, 400)
curvo = pd.DataFrame({"x": x_curvo, "y": y_curvo})
sns.regplot(data=curvo, x="x", y="y", scatter_kws={"alpha": 0.45, "s": 18},
            line_kws={"color": "#c0392b"}, ax=ax)
correlacion_curva = float(curvo.corr().iloc[0, 1])
ax.set_title(f"El aviso: regplot sobre una parábola\n"
             f"correlación = {correlacion_curva:.3f}, y la recta es plana",
             fontweight="bold", color="#922b21")
ax.set_xlabel("x")
ax.set_ylabel("y")

fig.tight_layout()
plt.show()

print(f"En el último panel, la correlación vale {correlacion_curva:+.3f} y regplot")
print("dibuja una recta prácticamente horizontal, como si no hubiera relación.")
print()
print("Y la relación es total: y = x² + ruido. La recta no está mal calculada; está")
print("contestando a la pregunta equivocada, porque una recta solo puede describir")
print("una relación lineal.")
print()
print("`regplot` admite `order=2` para ajustar un polinomio, y `lowess=True` para una")
print("curva suave sin suponer forma. Pero la decisión es tuya: Seaborn dibuja la")
print("recta sin avisar de que no procede.")

In [ ]:
# La misma parábola, con las tres opciones de regplot.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, (opciones, titulo) in zip(axes, [
        ({}, "recta (por defecto)"),
        ({"order": 2}, "order=2: polinomio de grado 2"),
        ({"lowess": True}, "lowess=True: curva suave, sin suponer forma")]):
    sns.regplot(data=curvo, x="x", y="y", scatter_kws={"alpha": 0.4, "s": 15},
                line_kws={"color": "#c0392b", "linewidth": 2.2}, ax=ax, **opciones)
    ax.set_title(titulo, fontweight="bold", fontsize=10)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.suptitle("Los mismos datos, tres ajustes", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

print("`lowess=True` es la opción prudente cuando no sabes la forma: no supone")
print("ninguna, sigue a los datos. El precio es que no da una ecuación, solo una")
print("curva, y que con pocos datos sigue al ruido.")

### 6.2 Series temporales con `lineplot`

`lineplot` hace algo que casi nadie nota: si hay **varias observaciones con el mismo
valor de X**, las agrega y dibuja la media con su banda de confianza. No dibuja los
datos, dibuja un resumen, y eso normalmente es lo que se quiere y a veces no.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
sns.lineplot(data=ventas, x="Mes", y="Importe", ax=ax, marker="o",
             errorbar=("ci", 95))
ax.set_title("Importe medio por mes, con intervalo de confianza\n"
             "lineplot ha agregado las ~400 transacciones de cada mes",
             fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Importe medio (€)")

ax = axes[1]
sns.lineplot(data=ventas, x="Mes", y="Importe", hue="Categoria", ax=ax,
             errorbar=None, marker="o", markersize=4)
ax.set_title("Lo mismo por categoría, sin banda (con seis series no se leería)",
             fontweight="bold")
ax.set_xlabel("Mes")
ax.set_ylabel("Importe medio (€)")
ax.legend(fontsize=8, title="Categoría", title_fontsize=9, ncol=2)

fig.tight_layout()
plt.show()

por_mes = ventas.groupby("Mes")["Importe"].agg(["size", "mean"])
print("Lo que `lineplot` ha hecho por su cuenta en el primer panel:")
print()
print(f"{'mes':>12} {'transacciones':>14} {'importe medio':>15}")
print("-" * 44)
for mes, fila in por_mes.iterrows():
    print(f"{mes:%Y-%m}      {int(fila['size']):>10}   {fila['mean']:>13.2f} €")
print()
print("Doce puntos en el gráfico, casi cinco mil filas en los datos. La banda es la")
print("incertidumbre sobre cada media mensual, y con ~400 observaciones por mes es")
print("estrecha. Si algún mes tuviera cuatro transacciones, su banda sería enorme, y")
print("eso es exactamente la información que hay que ver.")

### 6.3 Mapas de calor y tablas pivote

`sns.heatmap` sobre el resultado de un `pivot_table` es la combinación que más
rendimiento da por línea escrita en todo Seaborn: dos variables categóricas en los
ejes y una magnitud en el color.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Izquierda: importe medio por categoría y región.
tabla = ventas.pivot_table(values="Importe", index="Categoria", columns="Region",
                           aggfunc="mean")
sns.heatmap(tabla, annot=True, fmt=".0f", cmap="rocket_r", linewidths=0.6,
            cbar_kws={"label": "Importe medio (€)"}, ax=axes[0])
axes[0].set_title("Importe medio por categoría y región\n"
                  "Secuencial: la magnitud tiene orden pero no centro",
                  fontweight="bold", fontsize=11)
axes[0].set_xlabel("")
axes[0].set_ylabel("")

# Derecha: la MISMA tabla como desviación respecto a la media global. Ahora sí hay
# centro —el cero, "igual que la media"— y por tanto toca paleta divergente.
desviacion = tabla - ventas["Importe"].mean()
maximo = float(np.abs(desviacion.to_numpy()).max())
sns.heatmap(desviacion, annot=True, fmt="+.0f", cmap="vlag", center=0,
            vmin=-maximo, vmax=maximo, linewidths=0.6,
            cbar_kws={"label": "Diferencia con la media global (€)"}, ax=axes[1])
axes[1].set_title("La misma tabla, como diferencia con la media global\n"
                  "Divergente y centrada en cero",
                  fontweight="bold", fontsize=11)
axes[1].set_xlabel("")
axes[1].set_ylabel("")

fig.tight_layout()
plt.show()

print(f"Media global del importe: {ventas['Importe'].mean():.2f} €")
print()
print("Los dos mapas llevan exactamente la misma información y NO se leen igual.")
print("El primero contesta '¿cuánto?'; el segundo, '¿por encima o por debajo de lo")
print("normal?'. La segunda pregunta es casi siempre la interesante, y necesita")
print("paleta divergente y `center=0`.")
print()
print("Y un detalle que importa: `vmin=-maximo, vmax=maximo` hace la escala simétrica.")
print("Sin eso, el blanco de la paleta no cae en el cero y los colores engañan.")

## 7. Muchas variables a la vez

### 7.1 `pairplot`

Todas las nubes de puntos de todos los pares de variables numéricas, más la
distribución de cada una en la diagonal. Es lo primero que se hace al recibir un
conjunto de datos nuevo.

Con una advertencia de escala: son n² paneles. Con seis variables son treinta y seis
paneles, que ya cuesta mirar; con veinte serían cuatrocientos. **Por encima de unas
ocho variables, `pairplot` deja de servir** y hay que elegir cuáles mirar, por
ejemplo con el mapa de calor de correlaciones.

In [ ]:
numericas = ["Precio_Unitario", "Cantidad", "Descuento_%", "Importe"]
inicio = time.perf_counter()
rejilla = sns.pairplot(ventas[numericas + ["Categoria"]].sample(
                           800, random_state=20262027),
                       hue="Categoria", diag_kind="kde", corner=True,
                       plot_kws={"alpha": 0.55, "s": 14, "edgecolor": "none"},
                       height=1.9)
rejilla.figure.suptitle("pairplot de las cuatro variables numéricas\n"
                        "corner=True quita el triángulo repetido",
                        fontweight="bold", fontsize=13, y=1.02)
plt.show()
print(f"Dibujado en {time.perf_counter() - inicio:.1f} segundos, "
      f"y son {len(numericas)}² = {len(numericas) ** 2} combinaciones "
      f"(la mitad, con corner=True).")
print()
print("`corner=True` es casi obligatorio: la mitad de arriba de un pairplot es la de")
print("abajo con los ejes cambiados, así que no aporta nada y duplica el tiempo.")

### 7.2 `FacetGrid`: el mismo gráfico, un panel por grupo

`FacetGrid` reparte los datos por una o dos variables categóricas y aplica **el mismo
gráfico** a cada trozo. Es el patrón que sustituye a un bucle con filtrado.

La condición que lo hace funcionar, y que hay que respetar: **todos los paneles
comparten la escala**. Es lo que permite compararlos. Si cada panel tuviera su propia
escala, la comparación visual sería falsa, y es la razón por la que `sharex` y
`sharey` están puestos por defecto. No los quites sin un motivo escrito.

In [ ]:
rejilla = sns.FacetGrid(ventas, col="Region", row=None, col_wrap=4,
                        height=2.6, aspect=1.25, hue="Region")
rejilla.map_dataframe(sns.histplot, x="Importe", bins=25)
media_global = ventas["Importe"].mean()
rejilla.map(plt.axvline, x=media_global, color="black", linestyle="--", linewidth=1.4)
rejilla.set_titles("{col_name}", fontweight="bold", size=10)
rejilla.set_axis_labels("Importe (€)", "Transacciones")
rejilla.set(xlim=(0, ventas["Importe"].quantile(0.98)))
rejilla.figure.suptitle(
    f"Reparto del importe por región · la raya es la media global "
    f"({media_global:.0f} €)",
    fontweight="bold", fontsize=13, y=1.03)
plt.show()

print("La línea vertical de referencia es la que hace útil el gráfico: sin ella hay")
print("siete histogramas parecidos; con ella se ve qué regiones están a la derecha")
print("de la media global y cuáles a la izquierda.")
print()
resumen_facet = (ventas.groupby("Region")["Importe"]
                 .agg(["size", "mean"]).sort_values("mean", ascending=False))
print(f"{'región':>22} {'transacciones':>14} {'importe medio':>15}")
print("-" * 54)
for region, fila in resumen_facet.iterrows():
    marca = "por encima" if fila["mean"] > media_global else "por debajo"
    print(f"{region:>22} {int(fila['size']):>14} {fila['mean']:>13.2f} €  {marca}")

### 7.3 `jointplot`: el centro y los márgenes

Una nube de puntos con la distribución de cada variable en el margen. Sirve para no
tener que decidir entre mirar la relación y mirar los repartos, y el tipo se elige
según cuántos puntos hay.

In [ ]:
for tipo, comentario in [("scatter", "pocos puntos"),
                         ("hex", "muchos puntos: agrega, como el hexbin del cuaderno 02"),
                         ("kde", "contornos de densidad")]:
    rejilla = sns.jointplot(data=ventas.sample(1500, random_state=20262027),
                            x="Precio_Unitario", y="Importe", kind=tipo, height=4.6)
    rejilla.figure.suptitle(f"jointplot(kind='{tipo}') — {comentario}",
                            fontweight="bold", fontsize=11, y=1.02)
    rejilla.set_axis_labels("Precio unitario (€)", "Importe (€)")
    plt.show()

## 8. Cuándo Seaborn y cuándo Matplotlib, medido

Toca la pregunta del criterio 1.d, y aquí se puede contestar con un número: **cuántas
líneas cuesta la misma figura** con cada biblioteca.

La figura: importe medio por categoría, con su intervalo de confianza, ordenado.

In [ ]:
resumen_cat = ventas.groupby("Categoria")["Importe"]

# --- Con Seaborn ---
codigo_seaborn = '''
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=ventas, x="Categoria", y="Importe", order=orden,
            hue="Categoria", legend=False, errorbar=("ci", 95),
            capsize=0.15, ax=ax)
'''.strip()

# --- Con Matplotlib puro ---
codigo_matplotlib = '''
fig, ax = plt.subplots(figsize=(8, 4))
medias, errores = [], []
for categoria in orden:
    grupo = ventas.loc[ventas["Categoria"] == categoria, "Importe"].to_numpy()
    medias.append(grupo.mean())
    # Intervalo de confianza por remuestreo: 1.000 muestras con reemplazo
    remuestras = rng.choice(grupo, size=(1000, len(grupo)), replace=True).mean(axis=1)
    bajo, alto = np.percentile(remuestras, [2.5, 97.5])
    errores.append([grupo.mean() - bajo, alto - grupo.mean()])
errores = np.array(errores).T
colores = plt.get_cmap("tab10")(np.arange(len(orden)))
ax.bar(range(len(orden)), medias, yerr=errores, capsize=5, color=colores)
ax.set_xticks(range(len(orden)))
ax.set_xticklabels(orden, rotation=60, ha="right")
'''.strip()

# Y ahora se ejecutan los dos, de verdad.
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)

inicio = time.perf_counter()
sns.barplot(data=ventas, x="Categoria", y="Importe", order=orden,
            hue="Categoria", legend=False, errorbar=("ci", 95),
            capsize=0.15, ax=axes[0])
t_seaborn = time.perf_counter() - inicio
axes[0].set_title("Seaborn", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Importe medio (€)")
axes[0].tick_params(axis="x", rotation=60, labelsize=8)

inicio = time.perf_counter()
medias, errores = [], []
for categoria in orden:
    grupo = ventas.loc[ventas["Categoria"] == categoria, "Importe"].to_numpy()
    medias.append(grupo.mean())
    remuestras = rng.choice(grupo, size=(1000, len(grupo)), replace=True).mean(axis=1)
    bajo, alto = np.percentile(remuestras, [2.5, 97.5])
    errores.append([grupo.mean() - bajo, alto - grupo.mean()])
errores_np = np.array(errores).T
colores = plt.get_cmap("tab10")(np.arange(len(orden)))
axes[1].bar(range(len(orden)), medias, yerr=errores_np, capsize=5, color=colores)
axes[1].set_xticks(range(len(orden)))
axes[1].set_xticklabels(orden, rotation=60, ha="right", fontsize=8)
t_matplotlib = time.perf_counter() - inicio
axes[1].set_title("Matplotlib puro, haciendo lo mismo a mano", fontweight="bold")
axes[1].set_xlabel("")

fig.suptitle("La misma figura con las dos bibliotecas", fontsize=14,
             fontweight="bold")
fig.tight_layout()
plt.show()

lineas_seaborn = len([l for l in codigo_seaborn.split("\n") if l.strip()])
lineas_matplotlib = len([l for l in codigo_matplotlib.split("\n")
                         if l.strip() and not l.strip().startswith("#")])

print(f"{'':>14} {'líneas':>8} {'segundos':>10}")
print("-" * 34)
print(f"{'Seaborn':>14} {lineas_seaborn:>8} {t_seaborn:>10.3f}")
print(f"{'Matplotlib':>14} {lineas_matplotlib:>8} {t_matplotlib:>10.3f}")
print(f"{'proporción':>14} {lineas_matplotlib / lineas_seaborn:>7.1f}× "
      f"{t_matplotlib / t_seaborn:>9.1f}×")
print()
print("Y lo importante no es la proporción de líneas: es QUÉ líneas son.")
print()
print("Las de Matplotlib incluyen el remuestreo para calcular el intervalo de")
print("confianza. Ahí hay tres decisiones estadísticas —cuántas remuestras, qué")
print("percentiles, con o sin reemplazo— y cada una es una oportunidad de")
print("equivocarse. Seaborn las toma por ti, bien, y las documenta.")
print()
print("El precio es que las toma SIN QUE TE ENTERES, que es lo que se ha visto en la")
print("sección 5.3. Por eso hay que saber cuáles son.")

### La regla, resumida

| Situación | Herramienta |
|---|---|
| Exploración: quiero ver el reparto, la relación, la comparación | **Seaborn** |
| Hay que repartir en paneles por una variable | **Seaborn** (`col=`, `row=`) |
| El gráfico lleva estadística estándar (cuartiles, densidad, IC, regresión) | **Seaborn** |
| Composición irregular de paneles | **Matplotlib** (`GridSpec`) |
| Anotaciones colocadas al milímetro | **Matplotlib**, sobre el `Axes` de Seaborn |
| Un tipo de gráfico que Seaborn no tiene | **Matplotlib** |
| Millones de puntos | **Matplotlib** con agregación, o algo distinto (cuaderno 04) |

Y la forma habitual de trabajar no es elegir una: es **empezar con Seaborn y
terminar el gráfico con Matplotlib** sobre el `Axes` que ha devuelto.

In [ ]:
# El patrón de trabajo real: Seaborn dibuja, Matplotlib remata.
fig, ax = plt.subplots(figsize=(12, 5))

# 1. Seaborn hace el trabajo estadístico y el dibujo.
sns.boxplot(data=ventas, x="Categoria", y="Importe", order=orden,
            hue="Categoria", legend=False, ax=ax)

# 2. Matplotlib remata lo que Seaborn no sabe que quieres.
ax.set_ylim(0, ventas["Importe"].quantile(0.98))
ax.set_title("Importe por transacción y categoría\n"
             "Informática concentra el gasto alto; vestible es la categoría barata",
             fontsize=13, fontweight="bold", pad=14)
ax.set_xlabel("")
ax.set_ylabel("Importe (€)")
ax.tick_params(axis="x", rotation=25)
ax.axhline(media_global, color="#c0392b", linestyle="--", linewidth=1.6)
ax.text(0.012, media_global + 12, f"Media global: {media_global:.0f} €",
        color="#c0392b", fontsize=9, fontweight="bold")

# La anotación sobre la categoría más cara, colocada a mano.
mas_caro = orden[-1]
ax.annotate(f"Mediana de {mas_caro}:\n"
            f"{ventas.loc[ventas['Categoria'] == mas_caro, 'Importe'].median():.0f} €",
            xy=(len(orden) - 1, ventas.loc[ventas["Categoria"] == mas_caro,
                                           "Importe"].median()),
            xytext=(len(orden) - 2.4, ventas["Importe"].quantile(0.97)),
            arrowprops=dict(arrowstyle="->", lw=1.6, color="#1a5276"),
            fontsize=9, color="#1a5276", fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="#eaf2f8", alpha=0.9))

for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)

fig.tight_layout()
plt.show()

print("Cinco líneas de Seaborn y ocho de Matplotlib. Ninguna de las dos habría")
print("hecho este gráfico sola en trece líneas.")

## Ejercicios

Con las condiciones de los cuadernos anteriores, y una nueva propia de Seaborn:

4. **Si el color no codifica una variable, no lleva paleta.** Y si codifica la misma
   variable del eje X, se escribe `hue=` con esa variable y `legend=False`.

Todos los ejercicios usan `ventas`, el `DataFrame` limpio de la sección 2.2.

### Ejercicio 1 (Básico): Tema, contexto y paleta

Dibuja el mismo gráfico —importe medio por región, en barras— **tres veces**, con
tres combinaciones de tema, contexto y paleta pensadas para tres destinos:

1. Una figura pequeña dentro de la memoria de la práctica.
2. Una diapositiva proyectada en clase.
3. Un póster.

Justifica cada elección en una línea. Usa `with sns.axes_style(...)` y
`with sns.plotting_context(...)`.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 2 (Básico): Nivel `Axes` o nivel figura

Para cada uno de estos cuatro encargos, di **qué función de Seaborn** usarías y de qué
nivel es, y escríbelo:

1. Una figura 2×2 con cuatro gráficos distintos de `ventas`.
2. Un histograma del importe, un panel por región.
3. Una caja del importe por categoría, con una línea de la media global encima.
4. La nube de puntos de precio frente a importe con las distribuciones al margen.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 3 (Intermedio): La función de distribución acumulada

El objetivo es contestar a esta pregunta: **¿en qué región se hacen las compras más
grandes?**

1. Respóndela con un `barplot` de la media por región.
2. Respóndela con un `boxplot`.
3. Respóndela con un `ecdfplot` con `hue="Region"`.

Después escribe: **¿dan las tres la misma respuesta?** Si no, ¿cuál te fías más y por
qué? Y localiza en el `ecdfplot` la mediana de cada región leyéndola del gráfico, sin
calcularla.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 4 (Intermedio): `errorbar` y lo que afirma

Con `ventas`, dibuja el importe medio por categoría cuatro veces, con las cuatro
opciones de `errorbar` de la sección 5.3.

Después, para cada una, escribe **la frase que se puede afirmar** mirando ese gráfico
y no más. Por ejemplo, para `("ci", 95)` la frase empieza por «la media de esta
categoría está, con un 95 % de confianza, entre...».

Y contesta: si tuvieras que enseñar **uno solo** a la dirección de la tienda, ¿cuál?

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 5 (Intermedio): Mapa de calor de dos formas

Construye la tabla pivote de **número de transacciones** por región (filas) y mes
(columnas), y dibújala como mapa de calor dos veces:

1. Con los recuentos en bruto y paleta secuencial.
2. Como porcentaje de desviación respecto a la media de cada región, con paleta
   divergente centrada en cero.

Responde: ¿en qué región y qué mes hubo una anomalía que solo se ve en el segundo?

In [ ]:
# TODO: Escribe tu código aquí
# Pista: para el porcentaje por filas,
#   (tabla.T / tabla.mean(axis=1) - 1).T * 100

### Ejercicio 6 (Avanzado): Un `FacetGrid` que responda a una pregunta

Pregunta: **¿el descuento hace que la gente compre más unidades?**

1. Monta un `FacetGrid` por categoría con la nube de `Descuento_%` frente a
   `Cantidad`.
2. Añade en cada panel una recta de regresión.
3. Añade una línea horizontal con la cantidad media de esa categoría.
4. Comprueba que **la escala es común** a todos los paneles, y explica por qué es
   imprescindible aquí.

Y la parte importante: **contesta la pregunta**, con la limitación correspondiente.
Fíjate en que estos datos no permiten contestarla del todo, y explica por qué.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 7 (Avanzado): Seaborn primero, Matplotlib después

Produce **una sola figura** de calidad de informe que responda a esta pregunta:
*¿dónde y en qué está el gasto de TechStore?*

Requisitos:

1. Composición irregular con `GridSpec`: un panel grande y tres pequeños.
2. El contenido de los cuatro paneles, con Seaborn.
3. El remate con Matplotlib: título con la conclusión, una anotación en el hallazgo
   principal, líneas de referencia y bordes limpios.
4. Paleta coherente en los cuatro paneles.
5. Exportada a PDF, midiendo lo que pesa.

El título de la figura tiene que ser **una frase con la conclusión**, no «Análisis de
ventas».

In [ ]:
# TODO: Escribe tu código aquí

## Resumen

1. **Seaborn se apoya en Matplotlib.** Devuelve `Axes`, y todo lo del cuaderno 01
   sigue aplicando encima.
2. **Nivel `Axes` frente a nivel figura.** Si creaste la figura con `plt.subplots`,
   usa `ax=`; si quieres paneles automáticos, no crees ninguna figura.
3. **Tema, contexto y paleta son tres decisiones distintas.** El contexto es el que
   más se olvida y depende de a qué distancia se va a mirar el gráfico.
4. **El color codifica una variable o no va.** Seaborn lo ha convertido en norma: sin
   `hue`, no hay `palette`.
5. **`ecdfplot` no tiene parámetros que ajustar** y por eso es la forma más honesta de
   comparar repartos entre grupos.
6. **Seaborn calcula estadística por ti sin avisar.** `barplot` dibuja la media y un
   intervalo de confianza *de la media*, que no es la dispersión de los datos.
7. **`regplot` dibuja una recta aunque no proceda.** La correlación de una parábola es
   cero y la recta sale plana: la decisión de la forma es tuya.
8. **Un mapa de calor de diferencias necesita paleta divergente y `center=0`**, y la
   escala simétrica.
9. **En un `FacetGrid` la escala es común**, y eso es lo que hace que los paneles se
   puedan comparar. No lo quites sin motivo escrito.
10. **Lo normal es Seaborn primero y Matplotlib después**, no elegir una de las dos.

## Para seguir

- [Tutorial oficial de Seaborn](https://seaborn.pydata.org/tutorial.html) — la parte
  de *Overview of seaborn plotting functions* es justo la sección 3 de este cuaderno,
  explicada por sus autores.
- [Galería de ejemplos](https://seaborn.pydata.org/examples/index.html)
- [Elegir el tipo de gráfico](https://www.data-to-viz.com/) — un árbol de decisión
  que va del tipo de datos al gráfico, con el código en Python y en R.

**Siguiente:** el cuaderno 04 pasa a Plotly, que es la respuesta a la única cosa que
ni Matplotlib ni Seaborn hacen: gráficos que el lector puede tocar.